In [1]:
import pandas as pd
import numpy as np
import copy

Chenqu explored the barcode file, with the Levenshtein and everything as in `N08`, and looked at the global distributions of fractions and absolute read counts of sequencing error looking barcodes. Upon inspection, she chose to remove all barcodes with 10 or fewer reads (that removes 99.5% of cases of "being within 2 Levenshtein of a more populous barcode"). Let's do so.

Redo `N08` processing for a few steps.

In [2]:
barcodes = {}

df = pd.read_csv("barcodes.csv", index_col=0)
#there's a freak row that has an empty barcode entry. skipping
df = df[~df["barcode"].isna()]
for bc, tf in zip(df["barcode"], df["target"]):
    #disqualify abnormally sized BCs. we're supposed to be 12bp, let's keep it between 10 and 14
    if len(bc)<10 or len(bc)>14:
        continue
    #split ENST_NAME to get the gene name
    tf = tf.split("_")[1]
    #count up accordingly
    if tf not in barcodes:
        barcodes[tf] = {}
    if bc not in barcodes[tf]:
        barcodes[tf][bc] = 0
    barcodes[tf][bc] += 1

In [3]:
for tf in barcodes:
    barcodes[tf] = pd.DataFrame(barcodes[tf], index=["reads"]).T.sort_values("reads", ascending=False)

Actually do the filtering, and check in on our old test buddy.

In [4]:
for tf in barcodes:
    barcodes[tf] = barcodes[tf][barcodes[tf]["reads"]>10]

In [5]:
barcodes["ZNF562"]

,reads
TCCAAGGCGGAT,1013
TCGTCGCTTGCG,271
GTTACTGGGAGC,232
GAACTGCATGCT,127
GATTGACGGATT,76


Go back to `N08` for a bit.

In [6]:
for tf in barcodes:
    barcodes[tf]["barcode"] = barcodes[tf].index.copy()
    barcodes[tf]["tf"] = tf

In [7]:
barcodes_combined = pd.concat(barcodes.values(), ignore_index=True)

Now let's do something else new and neat - for barcodes with matches to multiple genes, let's pull them off into a separate data frame.

In [8]:
barcodes_multihit = barcodes_combined[barcodes_combined["barcode"].duplicated(keep=False)].copy(deep=True)
barcodes_combined = barcodes_combined[~barcodes_combined["barcode"].duplicated(keep=False)]

Let's check in on our test friend, and the barcode that gets lost as it has multiple hits.

In [9]:
barcodes_combined[barcodes_combined["tf"] == "ZNF562"]

,reads,barcode,tf
16,1013,TCCAAGGCGGAT,ZNF562
17,271,TCGTCGCTTGCG,ZNF562
18,232,GTTACTGGGAGC,ZNF562
19,127,GAACTGCATGCT,ZNF562


In [10]:
barcodes_multihit[barcodes_multihit["barcode"] == "GATTGACGGATT"]

,reads,barcode,tf
20,76,GATTGACGGATT,ZNF562
13399,297,GATTGACGGATT,KLF4
22701,325,GATTGACGGATT,GRHL3


Create entries for the multihits matching the formatting of the original data frame.

In [11]:
grouped = barcodes_multihit.groupby('barcode')['reads'].sum().reset_index()
grouped["tf"] = "MULTIHIT"
grouped = grouped[["reads", "barcode", "tf"]]

Check in on our test barcode.

In [12]:
grouped[grouped["barcode"] == "GATTGACGGATT"]

,reads,barcode,tf
586,698,GATTGACGGATT,MULTIHIT


Glue it all together.

In [13]:
barcodes_combined = pd.concat([barcodes_combined, grouped], axis=0).reset_index(drop=True)
barcodes_combined

,reads,barcode,tf
0,1282,AGTTATAGTTGG,OTX2
1,717,ATCGCGATTCGG,OTX2
2,485,CTCACGGCGGTG,OTX2
3,452,TCCGTAATTCAA,OTX2
4,386,CACGCTAATCAC,OTX2
...,...,...,...
23273,370,TTTTGACATCCG,MULTIHIT
23274,861,TTTTGTGTAACG,MULTIHIT
23275,33,TTTTGTTTATCG,MULTIHIT
23276,1608,TTTTTAGGTTAT,MULTIHIT


Check on ZNF562 and the multi-hit.

In [14]:
barcodes_combined[barcodes_combined["tf"] == "ZNF562"]

,reads,barcode,tf
15,1013,TCCAAGGCGGAT,ZNF562
16,271,TCGTCGCTTGCG,ZNF562
17,232,GTTACTGGGAGC,ZNF562
18,127,GAACTGCATGCT,ZNF562


In [15]:
barcodes_combined[barcodes_combined["barcode"] == "GATTGACGGATT"]

,reads,barcode,tf
22665,698,GATTGACGGATT,MULTIHIT


Think we're okay. Dunk it out.

In [16]:
read = "R2"
#post-barcode sequence in revcomp, BC, pre-barcode sequence in revcomp
pattern = "CCGGAGCGAGACAAG(BC)GGTCGCTTCGGCGCG"
feature_type = "Antibody Capture"

def revcomp(seq):
    return seq.translate(str.maketrans('ATGC','TACG'))[::-1]

with open("feature_ref_guideseq_v2.csv", "w") as fid:
    fid.writelines(["id,name,read,pattern,sequence,feature_type\n"])
    for ind in barcodes_combined.index:
        #we need to revcomp the bc
        rbc = revcomp(barcodes_combined.loc[ind, "barcode"])
        #create a naming scheme - gene-bc-counts
        name = barcodes_combined.loc[ind, "tf"]+'-'+rbc+'-'+str(barcodes_combined.loc[ind, "reads"])
        fid.writelines(",".join([name,name,read,pattern,rbc,feature_type])+"\n")

In [17]:
#the multihit df can just be dunked to a csv
barcodes_multihit.to_csv("feature_ref_guideseq_v2_multihit.csv")